# 第 6 课：因果动态置信度

目标：只有预测区间真值已经完整可见时，才根据效果更新记忆置信度。

In [ ]:
import sys
from pathlib import Path

# 同时兼容：从仓库根目录启动 Jupyter，或从 notebooks/ 目录启动。
search_starts = [Path.cwd(), *Path.cwd().parents]
repo_root = next((path for path in search_starts if (path / "pyproject.toml").exists()), None)
if repo_root is None:
    raise RuntimeError("没有找到 pyproject.toml；请从仓库目录启动 Jupyter。")

src_dir = repo_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

print(f"仓库根目录: {repo_root}")
print(f"Python: {sys.version.split()[0]}")

## 6.1 创建一条延迟反馈事件

核心源码：[confidence.py](../src/memcast_uav/confidence.py)  
文字讲解：[06_confidence.md](../tutorial/06_confidence.md)

In [ ]:
from memcast_uav.confidence import CausalConfidenceQueue, FeedbackEvent
from memcast_uav.data import make_synthetic_flight, make_train_test_windows
from memcast_uav.memory import build_memory

flight = make_synthetic_flight()
train, test = make_train_test_windows(flight, split_index=504)
memory = build_memory(train, limit=3)
target = memory.entries[0]

event = FeedbackEvent(
    event_id="notebook-feedback",
    available_at=test[0].forecast_end,
    contributing_ids=[target.entry_id],
    feedback={"success": True},
)
queue = CausalConfidenceQueue()
queue.add(event)
print("反馈成熟时刻:", event.available_at)

## 6.2 预测刚开始：未来真值还不可见

In [ ]:
before = target.confidence
early_updates = queue.apply_available(test[0].forecast_start, memory)
print("更新条目数:", early_updates)
print("置信度:", before, "→", target.confidence)
assert early_updates == 0
assert target.confidence == before

## 6.3 预测区间结束：反馈可以应用

In [ ]:
mature_updates = queue.apply_available(test[0].forecast_end, memory)
after = target.confidence
print("更新条目数:", mature_updates)
print("置信度:", before, "→", after)
assert mature_updates == 1
assert after > before

## 6.4 同一事件不能重复应用

In [ ]:
repeated_updates = queue.apply_available(test[0].forecast_end, memory)
print("重复调用更新条目数:", repeated_updates)
print("置信度仍为:", target.confidence)
assert repeated_updates == 0

## 6.5 分块练习：失败事件不加分

In [ ]:
failed_target = memory.entries[1]
failed_event = FeedbackEvent(
    event_id="notebook-failed-feedback",
    available_at=test[0].forecast_end,
    contributing_ids=[failed_target.entry_id],
    feedback={"success": False},
)
failed_queue = CausalConfidenceQueue()
failed_queue.add(failed_event)
failed_queue.apply_available(test[0].forecast_end, memory)
print("失败经验置信度:", failed_target.confidence)
assert failed_target.confidence == 0.0

## 6.6 本课验收

In [ ]:
import subprocess

completed = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_confidence.py", "-q"],
    cwd=repo_root,
    check=True,
    text=True,
    capture_output=True,
)
print(completed.stdout)

[← 第 5 课](05_reflection.ipynb) · [教程目录](README.md) · [下一课：端到端管线 →](07_end_to_end.ipynb)